In [7]:
import torch 
g = torch.Generator().manual_seed(2147483647)

In [8]:
class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn(fan_in, fan_out, generator=g) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None 
    
    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out 
    
    def parameters(self):
        return [self.weight] + [[] if self.bias is None else self.bias]


class BatchNorm:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        # constants 
        self.eps = eps
        self.momentum = momentum
        self.training = True 
        # trainable parameters 
        self.gamma = torch.ones(dim) # scale
        self.beta = torch.zeros(dim) # shift
        # running mean and var
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        # forward pass 
        if self.training: 
            xmean = x.mean(0, keepdim=True) # batch mean 
            xvar = x.var(0, keepdim=True) # batch variance 
        else:
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # standardize normalization 
        self.out = self.gamma * xhat + self.beta # scale and shift 
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean 
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar 

        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]

class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    def parameters(self): 
        return [] # tanh has no parameters 

In [11]:
n_embd = 10 # dimensionality of the character embedding vectors 
n_hidden = 100 # number of neurons at the hidden layers of the mlp 
vocab_size = 27
block_size = 3

C = torch.randn(vocab_size, n_embd, generator=g)

layers = [
    Linear(block_size * n_embd, n_hidden), Tanh(),
    Linear(             n_hidden, n_hidden), Tanh(),
    Linear(             n_hidden, n_hidden), Tanh(),
    Linear(             n_hidden, n_hidden), Tanh(),
    Linear(             n_hidden, n_hidden), Tanh(),
    Linear(             n_hidden, vocab_size),
]


# DOn't keep track of compututional graph of these init operations - hence require no grad
with torch.no_grad(): # we 
    # scale down the weights of the last layer so that we have a uniform weight dist
    layers[-1].weight *= 0.1 # it prevents overconfident predictions leading to a very high loss in eary training steps 
    for layer in layers[:-1]: # for the remaining layers 
        if isinstance(layer, Linear): # if the layer is a linear layer 
            layer.weight *= 5/3 # use the kaiming init - the divison by fan_in**0.5 has been added to layer.weight in Linear class


parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters)) # total number of parameter 

for p in parameters:
    p.requires_grad = True 

46497
